In [1]:
import torch
import torch.nn as nn
import torchvision.transforms as transforms
from torchvision import datasets
from torch.utils.data import DataLoader
import timm

# Much stronger augmentation
train_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(30),
    transforms.ColorJitter(
        brightness=0.4,
        contrast=0.4,
        saturation=0.4,
        hue=0.1
    ),
    transforms.RandomGrayscale(p=0.1),
    transforms.ToTensor(),
    transforms.Normalize(
        [0.485, 0.456, 0.406],
        [0.229, 0.224, 0.225])
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        [0.485, 0.456, 0.406],
        [0.229, 0.224, 0.225])
])

DATA_PATH = "../data/Indian Food Images/Indian Food Images"

full_dataset = datasets.ImageFolder(
    DATA_PATH, transform=train_transform)
class_names = full_dataset.classes
num_classes = len(class_names)

train_size = int(0.8 * len(full_dataset))
val_size = len(full_dataset) - train_size
train_set, val_set = torch.utils.data.random_split(
    full_dataset, [train_size, val_size])
val_set.dataset.transform = val_transform

train_loader = DataLoader(
    train_set, batch_size=32, shuffle=True)
val_loader = DataLoader(
    val_set, batch_size=32, shuffle=False)

# EfficientNet B2 — much better than MobileNetV2
model = timm.create_model(
    'efficientnet_b2',
    pretrained=True,
    num_classes=num_classes
)
device = torch.device('cpu')
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(
    model.parameters(), lr=0.0005)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=20)

best_acc = 0
for epoch in range(20):
    model.train()
    running_loss = 0
    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()

    model.eval()
    correct = total = 0
    with torch.no_grad():
        for images, labels in val_loader:
            images = images.to(device)
            labels = labels.to(device)
            outputs = model(images)
            _, predicted = outputs.max(1)
            correct += predicted.eq(labels).sum().item()
            total += labels.size(0)

    acc = 100 * correct / total
    scheduler.step()
    print(f"Epoch {epoch+1}/20 | "
          f"Loss: {running_loss/len(train_loader):.3f} | "
          f"Val Acc: {acc:.1f}%")

    if acc > best_acc:
        best_acc = acc
        torch.save({
            'model_state': model.state_dict(),
            'class_names': class_names,
            'model_name': 'efficientnet_b2'
        }, "../models/nutrivision_v2.pth")
        print(f"  ✅ Saved! Best: {best_acc:.1f}%")

print(f"Training complete! Best: {best_acc:.1f}%")

model.safetensors:   0%|          | 0.00/36.8M [00:00<?, ?B/s]

c:\Users\sujal\OneDrive\Desktop\Nutrivision-India\venv\Lib\site-packages\huggingface_hub\file_download.py:137: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\sujal\.cache\huggingface\hub\models--timm--efficientnet_b2.ra_in1k. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


Epoch 1/20 | Loss: 2.781 | Val Acc: 54.9%
  ✅ Saved! Best: 54.9%
Epoch 2/20 | Loss: 0.634 | Val Acc: 61.9%
  ✅ Saved! Best: 61.9%
Epoch 3/20 | Loss: 0.161 | Val Acc: 66.9%
  ✅ Saved! Best: 66.9%
Epoch 4/20 | Loss: 0.067 | Val Acc: 67.8%
  ✅ Saved! Best: 67.8%
Epoch 5/20 | Loss: 0.040 | Val Acc: 69.6%
  ✅ Saved! Best: 69.6%
Epoch 6/20 | Loss: 0.027 | Val Acc: 70.2%
  ✅ Saved! Best: 70.2%
Epoch 7/20 | Loss: 0.019 | Val Acc: 70.1%
Epoch 8/20 | Loss: 0.022 | Val Acc: 69.0%
Epoch 9/20 | Loss: 0.018 | Val Acc: 70.0%
Epoch 10/20 | Loss: 0.013 | Val Acc: 69.9%
Epoch 11/20 | Loss: 0.013 | Val Acc: 69.4%
Epoch 12/20 | Loss: 0.012 | Val Acc: 69.2%
Epoch 13/20 | Loss: 0.011 | Val Acc: 70.1%
Epoch 14/20 | Loss: 0.011 | Val Acc: 70.4%
  ✅ Saved! Best: 70.4%
Epoch 15/20 | Loss: 0.010 | Val Acc: 70.6%
  ✅ Saved! Best: 70.6%
Epoch 16/20 | Loss: 0.010 | Val Acc: 70.4%
Epoch 17/20 | Loss: 0.009 | Val Acc: 70.5%
Epoch 18/20 | Loss: 0.009 | Val Acc: 70.4%
Epoch 19/20 | Loss: 0.009 | Val Acc: 71.0%
  ✅ Save